In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
import rrcf
import numpy as np
import pandas as pd

def rcf_scores_stream(
    X: np.ndarray,
    num_trees: int = 40,
    tree_size: int = 256,
    random_state: int = 42
) -> np.ndarray:
    rng = np.random.default_rng(random_state)

    forest = [rrcf.RCTree() for _ in range(num_trees)]
    scores = np.zeros(len(X), dtype=float)

    for i in range(len(X)):
        x_i = X[i]
        s = 0.0
        for tree in forest:
            old_i = i - tree_size
            if old_i in tree.leaves:
                tree.forget_point(old_i)

            tree.insert_point(x_i, index=i)
            s += tree.codisp(i)
        scores[i] = s

    med = np.median(scores[np.isfinite(scores)]) if np.any(np.isfinite(scores)) else 1.0
    if med == 0:
        med = 1.0
    return scores / med

In [ ]:
from xgboost import XGBClassifier
from preprocessing.preprocess import prep, append_results
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics

def train_xgb_ttp_with_rcf_filter(
    df: pd.DataFrame,
    train_size: int,
    test_size: int,
    step: int,
    rcf_quantile: float = 0.95,
    rcf_num_trees: int = 40,
    rcf_tree_size: int = 256,
):
    scale_cols = [
        "Open","High","Low","Close",
        "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
        "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
    ]

    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12, 24, 48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes": 3},
        scale_cols=scale_cols
    )

    for X_train, X_test, y_train, y_test, scaler, train_idx, test_idx in splitter:

        # строим общий массив для RCF: train+test (внимание: уже scaled)
        X_chunk = np.vstack([X_train.values, X_test.values]).astype(np.float32)

        scores = rcf_scores_stream(
            X_chunk,
            num_trees=rcf_num_trees,
            tree_size=rcf_tree_size,
            random_state=42
        )

        train_scores = scores[:len(X_train)]
        test_scores  = scores[len(X_train):]

        thr = float(np.quantile(train_scores, rcf_quantile))

        tr_mask = train_scores <= thr
        te_mask = test_scores <= thr

        X_train_f = X_train.values[tr_mask]
        y_train_f = y_train.values[tr_mask]

        X_test_f = X_test.values[te_mask]
        y_test_f = y_test.values[te_mask]

        # защита от “слишком агрессивного фильтра”
        if len(y_train_f) < 100 or len(y_test_f) < 30:
            continue

        model = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=42
        )

        model.fit(X_train_f, y_train_f)
        y_pred = model.predict(X_test_f)

        metrics = merged_metrics(y_test_f, y_pred)

        append_results({
            "task_type": "classification",
            "model_name": "XGBoost",
            "model_family": "xgb",
            "model_params": {
                "n_estimators": 300,
                "max_depth": 5,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,

                # RCF filter
                "rcf_num_trees": rcf_num_trees,
                "rcf_tree_size": rcf_tree_size,
                "rcf_quantile": rcf_quantile,
                "rcf_threshold": thr,
            },
            "target_name": "ttp",
            "target_variant": "3class",
            "horizons": "12_24_48",

            "train_kept": int(tr_mask.sum()),
            "train_dropped": int((~tr_mask).sum()),
            "test_kept": int(te_mask.sum()),
            "test_dropped": int((~te_mask).sum()),

            **metrics
        })

In [ ]:
name = "AFKS"
df = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

train_xgb_ttp_with_rcf_filter(
    df=df,
    train_size=1000,
    test_size=200,
    step=100,
    rcf_quantile=0.95,
    rcf_num_trees=40,
    rcf_tree_size=256
)